
Assignments


In [0]:
cust_raw_df=spark.read.csv("/Volumes/wh_gb_test/gb_test/my_volume/cust_data/customer_data.csv",header=True,sep=",",inferSchema=True)
trans_raw_df=spark.read.csv("/Volumes/wh_gb_test/gb_test/my_volume/cust_data/sales_data.csv",header=True,sep=",",inferSchema=True)
cust_raw_df.show(5)
trans_raw_df.show(5)

trans_details_df=trans_raw_df.join(cust_raw_df,trans_raw_df.customer_id==cust_raw_df.customer_id)
trans_details_df.show(5)

#Are there any transactions with a customer_id that does not exist in the customer master table? Identify and count orphaned transactions.
orphaned_txn_df = trans_raw_df.join(
    cust_raw_df,
    trans_raw_df.customer_id == cust_raw_df.customer_id,
    how="left_anti"
)
display(orphaned_txn_df)
print(f"Orphaned transactions count: {orphaned_txn_df.count()}")

In [0]:

trans_cloth_df=trans_details_df.filter("category='Clothing'")
trans_cloth_df.describe
trans_cloth_df.createOrReplaceTempView("trans_cloth")

revenue_df = (trans_cloth_df
    .selectExpr(
        "(date_format(invoice_date, 'MMMM') || ' ' || date_format(invoice_date, 'yyyy')) as Month",
        "quantity * price as item_revenue"
    )
    .groupBy("Month")
    .agg(F.sum("item_revenue").alias("Revenue"))
)
display(revenue_df)

In [0]:
from pyspark.sql import functions as F

trans_details_df=trans_raw_df.join(cust_raw_df,trans_raw_df.customer_id==cust_raw_df.customer_id).drop(cust_raw_df.customer_id)
trans_details_df.show(5)

top_customers_df = (trans_details_df
    .selectExpr("customer_id", "quantity * price as item_total")
    .groupBy("customer_id")
    .agg(F.sum("item_total").alias("total_spent"))
    .orderBy(F.col("total_spent").desc())
    .limit(5))
top_customers_df.show()

In [0]:
#For each customer, show their total revenue and classify them as 'High Value' (≥ 5000), 'Mid Value' (≥ 1000), or 'Low Value' (< 1000)

customers_rev_df = (trans_details_df
    .selectExpr("customer_id", "quantity * price as item_total")
    .groupBy("customer_id")
    .agg(F.sum("item_total").alias("total_spent")))
customers_rev_df.display()

customers_segment_df=customers_rev_df.selectExpr(
        "customer_id",
        "total_spent",
        "case when total_spent >= 5000 then 'High Value' when total_spent >= 1000 then 'Mid Value' else 'Low Value' end as customer_segment"
    )
customers_segment_df.display()

In [0]:
# Usecase 5: Spending difference by gender
gender_spending_df = (
    trans_details_df
    .groupBy("gender")
    .agg(
        F.avg(F.col("quantity") * F.col("price")).alias("avg_transaction_value"),
        F.sum(F.col("quantity") * F.col("price")).alias("total_revenue"),
        F.count("*").alias("transaction_count")
    )
)
display(gender_spending_df)

# Usecase 6: Most popular payment method per product category
payment_category_df = (
    trans_details_df
    .groupBy("category", "payment_method")
    .agg(F.count("*").alias("transaction_count"))
    .orderBy("category", F.col("transaction_count").desc())
)
display(payment_category_df)

In [0]:


# Usecase 8: Revenue by age group
from pyspark.sql import functions as F

trans_age_df = trans_details_df.withColumn(
    "age_group",
    F.when(F.col("age") < 20, "Teens")
     .when((F.col("age") >= 20) & (F.col("age") <= 35), "Young Adults")
     .when((F.col("age") >= 36) & (F.col("age") <= 50), "Adults")
     .when(F.col("age") > 50, "Seniors")
     .otherwise("Unknown")
)
age_group_revenue_df = (trans_age_df
    .groupBy("age_group")
    .agg(F.sum(F.col("quantity") * F.col("price")).alias("total_revenue"))
    .orderBy(F.col("total_revenue").desc())
)
display(age_group_revenue_df)

# Usecase 9: Customers with last purchase > 90 days before max invoice date
max_date = trans_details_df.agg(F.max("invoice_date")).first()[0]
recent_purchase_df = (
    trans_details_df
    .groupBy("customer_id")
    .agg(F.max("invoice_date").alias("last_purchase_date"))
    .withColumn("days_since_last", F.datediff(F.lit(max_date), F.col("last_purchase_date")))
    .filter(F.col("days_since_last") > 90)
)
display(recent_purchase_df)

# Usecase 10: Clean master dataset with derived columns and persist as Parquet
master_df = (
    trans_raw_df.join(cust_raw_df, "customer_id", "left")
    .withColumn("revenue", F.col("quantity") * F.col("price"))
    .withColumn("month", F.date_format("invoice_date", "MMMM"))
    .withColumn("day_of_week", F.date_format("invoice_date", "EEEE"))
    .withColumn(
        "age_group",
        F.when(F.col("age") < 20, "Teens")
         .when((F.col("age") >= 20) & (F.col("age") <= 35), "Young Adults")
         .when((F.col("age") >= 36) & (F.col("age") <= 50), "Adults")
         .when(F.col("age") > 50, "Seniors")
         .otherwise("Unknown")
    )
    .na.fill({"quantity": 0, "price": 0, "revenue": 0, "age": -1, "gender": "Unknown", "category": "Unknown", "payment_method": "Unknown"})
)
master_df.write.mode("overwrite").parquet("/Volumes/wh_gb_test/gb_test/my_volume/analysis/master_dataset")
display(master_df)

# Usecase 11: Gender distribution across product categories
gender_category_df = (
    trans_details_df
    .groupBy("category", "gender")
    .agg(F.count("*").alias("transaction_count"))
    .orderBy("category", "gender")
)
display(gender_category_df)

# Usecase 12: Total revenue in 2022
revenue_2022_df = (
    trans_details_df
    .filter(F.year("invoice_date") == 2022)
    .agg(F.sum(F.col("quantity") * F.col("price")).alias("total_revenue_2022"))
)
display(revenue_2022_df)